# Polars Comprehensive Tutorial

A hands-on guide from fundamentals to advanced Polars operations.

## Table of Contents

1. [Setup & Introduction](#1-setup--introduction)
2. [Creating DataFrames & Series](#2-creating-dataframes--series)
3. [Schema & Data Types](#3-schema--data-types)
4. [Basic DataFrame Operations](#4-basic-dataframe-operations)
5. [Expressions & Column Operations](#5-expressions--column-operations)
6. [Aggregations](#6-aggregations)
7. [Joins](#7-joins)
8. [Window Functions](#8-window-functions)
9. [Nested Data & Structs](#9-nested-data--structs)
10. [Lazy Evaluation & Query Optimization](#10-lazy-evaluation--query-optimization)
11. [Reading & Writing Data](#11-reading--writing-data)
12. [String & Temporal Operations](#12-string--temporal-operations)
13. [Missing Data](#13-missing-data)
14. [Reshaping (Pivot, Unpivot, Explode)](#14-reshaping-pivot-unpivot-explode)
15. [Performance Tips & Patterns](#15-performance-tips--patterns)

---
## 1. Setup & Introduction

Polars is a blazingly fast DataFrame library written in Rust with a Python API. Key advantages:
- **Multi-threaded** — uses all CPU cores by default
- **Lazy evaluation** — query optimization before execution
- **Apache Arrow memory** — zero-copy interop, cache-friendly
- **Expressive API** — composable expressions, no index

In [39]:
from datetime import date, timedelta

import numpy as np

import polars as pl

print(f"Polars version: {pl.__version__}")
pl.Config.set_tbl_rows(15)
pl.Config.set_fmt_str_lengths(50)

Polars version: 1.43.2


polars.config.Config

---
## 2. Creating DataFrames & Series

Polars has two main data structures:
- **Series** — a single typed column
- **DataFrame** — a collection of named Series

In [40]:
# Series — a single typed column
s = pl.Series("temperatures", [72.5, 68.1, 75.3, 80.0, 65.9])
print(s)
print(f"\nDtype: {s.dtype}, Length: {s.len()}, Mean: {s.mean():.1f}")

shape: (5,)
Series: 'temperatures' [f64]
[
	72.5
	68.1
	75.3
	80.0
	65.9
]

Dtype: Float64, Length: 5, Mean: 72.4


In [41]:
# DataFrame from dict (most common)
df_simple = pl.DataFrame({
    "name": ["Alice", "Bob", "Charlie"],
    "age": [30, 25, 35],
    "city": ["NYC", "LA", "Chicago"],
})
print(df_simple)

shape: (3, 3)
┌─────────┬─────┬─────────┐
│ name    ┆ age ┆ city    │
│ ---     ┆ --- ┆ ---     │
│ str     ┆ i64 ┆ str     │
╞═════════╪═════╪═════════╡
│ Alice   ┆ 30  ┆ NYC     │
│ Bob     ┆ 25  ┆ LA      │
│ Charlie ┆ 35  ┆ Chicago │
└─────────┴─────┴─────────┘


In [42]:
# --- E-COMMERCE DATASET (used throughout) ---

customers = pl.DataFrame({
    "customer_id": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10],
    "name": [
        "Alice Johnson",
        "Bob Smith",
        "Charlie Brown",
        "Diana Ross",
        "Eve Wilson",
        "Frank Miller",
        "Grace Lee",
        "Henry Davis",
        "Ivy Chen",
        "Jack Thompson",
    ],
    "email": [
        "alice@email.com",
        "bob@email.com",
        "charlie@email.com",
        "diana@email.com",
        "eve@email.com",
        "frank@email.com",
        "grace@email.com",
        "henry@email.com",
        "ivy@email.com",
        "jack@email.com",
    ],
    "city": [
        "New York",
        "Los Angeles",
        "Chicago",
        "Houston",
        "Phoenix",
        "New York",
        "San Francisco",
        "Chicago",
        "Seattle",
        "Boston",
    ],
    "state": ["NY", "CA", "IL", "TX", "AZ", "NY", "CA", "IL", "WA", "MA"],
    "signup_date": [
        date(2020, 1, 15),
        date(2020, 3, 22),
        date(2020, 6, 10),
        date(2021, 1, 5),
        date(2021, 4, 18),
        date(2021, 7, 30),
        date(2021, 9, 12),
        date(2022, 2, 28),
        date(2022, 5, 14),
        date(2022, 8, 1),
    ],
})

print("=== CUSTOMERS ===")
print(customers)

=== CUSTOMERS ===
shape: (10, 6)
┌─────────────┬───────────────┬───────────────────┬───────────────┬───────┬─────────────┐
│ customer_id ┆ name          ┆ email             ┆ city          ┆ state ┆ signup_date │
│ ---         ┆ ---           ┆ ---               ┆ ---           ┆ ---   ┆ ---         │
│ i64         ┆ str           ┆ str               ┆ str           ┆ str   ┆ date        │
╞═════════════╪═══════════════╪═══════════════════╪═══════════════╪═══════╪═════════════╡
│ 1           ┆ Alice Johnson ┆ alice@email.com   ┆ New York      ┆ NY    ┆ 2020-01-15  │
│ 2           ┆ Bob Smith     ┆ bob@email.com     ┆ Los Angeles   ┆ CA    ┆ 2020-03-22  │
│ 3           ┆ Charlie Brown ┆ charlie@email.com ┆ Chicago       ┆ IL    ┆ 2020-06-10  │
│ 4           ┆ Diana Ross    ┆ diana@email.com   ┆ Houston       ┆ TX    ┆ 2021-01-05  │
│ 5           ┆ Eve Wilson    ┆ eve@email.com     ┆ Phoenix       ┆ AZ    ┆ 2021-04-18  │
│ 6           ┆ Frank Miller  ┆ frank@email.com   ┆ New York      ┆

In [43]:
products = pl.DataFrame({
    "product_id": [
        101,
        102,
        103,
        104,
        105,
        106,
        107,
        108,
        109,
        110,
        111,
        112,
        113,
        114,
        115,
    ],
    "product_name": [
        "Laptop Pro 15",
        "Wireless Mouse",
        "USB-C Hub",
        "Standing Desk",
        "Ergonomic Chair",
        'Monitor 27"',
        "Mechanical Keyboard",
        "Desk Lamp",
        "Webcam HD",
        "Noise Cancelling Headphones",
        "Bookshelf",
        "Whiteboard",
        "Notebook Set",
        "Pen Set",
        "Cable Management Kit",
    ],
    "category": [
        "Electronics",
        "Electronics",
        "Electronics",
        "Furniture",
        "Furniture",
        "Electronics",
        "Electronics",
        "Furniture",
        "Electronics",
        "Electronics",
        "Furniture",
        "Office Supplies",
        "Office Supplies",
        "Office Supplies",
        "Office Supplies",
    ],
    "price": [
        1299.99,
        29.99,
        49.99,
        599.99,
        449.99,
        399.99,
        89.99,
        39.99,
        69.99,
        249.99,
        129.99,
        79.99,
        12.99,
        8.99,
        19.99,
    ],
    "stock_quantity": [
        50,
        200,
        150,
        30,
        45,
        75,
        120,
        200,
        100,
        80,
        60,
        40,
        500,
        800,
        300,
    ],
})

print("=== PRODUCTS ===")
print(products)

=== PRODUCTS ===
shape: (15, 5)
┌────────────┬─────────────────────────────┬─────────────────┬─────────┬────────────────┐
│ product_id ┆ product_name                ┆ category        ┆ price   ┆ stock_quantity │
│ ---        ┆ ---                         ┆ ---             ┆ ---     ┆ ---            │
│ i64        ┆ str                         ┆ str             ┆ f64     ┆ i64            │
╞════════════╪═════════════════════════════╪═════════════════╪═════════╪════════════════╡
│ 101        ┆ Laptop Pro 15               ┆ Electronics     ┆ 1299.99 ┆ 50             │
│ 102        ┆ Wireless Mouse              ┆ Electronics     ┆ 29.99   ┆ 200            │
│ 103        ┆ USB-C Hub                   ┆ Electronics     ┆ 49.99   ┆ 150            │
│ 104        ┆ Standing Desk               ┆ Furniture       ┆ 599.99  ┆ 30             │
│ 105        ┆ Ergonomic Chair             ┆ Furniture       ┆ 449.99  ┆ 45             │
│ 106        ┆ Monitor 27"                 ┆ Electronics     ┆ 399.9

In [44]:
orders = pl.DataFrame({
    "order_id": list(range(1001, 1026)),
    "customer_id": [
        1,
        2,
        1,
        3,
        4,
        5,
        2,
        6,
        3,
        7,
        1,
        8,
        4,
        9,
        5,
        10,
        2,
        6,
        7,
        3,
        1,
        4,
        9,
        8,
        5,
    ],
    "order_date": [
        date(2023, 1, 10),
        date(2023, 1, 15),
        date(2023, 2, 20),
        date(2023, 3, 5),
        date(2023, 3, 18),
        date(2023, 4, 2),
        date(2023, 4, 22),
        date(2023, 5, 10),
        date(2023, 5, 28),
        date(2023, 6, 15),
        date(2023, 7, 1),
        date(2023, 7, 20),
        date(2023, 8, 5),
        date(2023, 8, 22),
        date(2023, 9, 10),
        date(2023, 9, 28),
        date(2023, 10, 15),
        date(2023, 10, 30),
        date(2023, 11, 12),
        date(2023, 11, 25),
        date(2023, 12, 5),
        date(2023, 12, 18),
        date(2023, 12, 28),
        date(2024, 1, 5),
        date(2024, 1, 20),
    ],
    "status": [
        "completed",
        "completed",
        "completed",
        "completed",
        "cancelled",
        "completed",
        "completed",
        "completed",
        "returned",
        "completed",
        "completed",
        "completed",
        "completed",
        "completed",
        "completed",
        "cancelled",
        "completed",
        "completed",
        "completed",
        "completed",
        "completed",
        "completed",
        "completed",
        "completed",
        "pending",
    ],
    "total_amount": [
        1349.98,
        599.99,
        539.97,
        1749.98,
        89.99,
        329.97,
        449.99,
        1299.99,
        249.99,
        869.97,
        399.99,
        159.98,
        1949.97,
        79.98,
        599.99,
        1299.99,
        269.98,
        489.98,
        149.98,
        2099.97,
        339.98,
        699.98,
        1349.98,
        449.99,
        89.99,
    ],
})

print("=== ORDERS ===")
print(orders)

=== ORDERS ===
shape: (25, 5)
┌──────────┬─────────────┬────────────┬───────────┬──────────────┐
│ order_id ┆ customer_id ┆ order_date ┆ status    ┆ total_amount │
│ ---      ┆ ---         ┆ ---        ┆ ---       ┆ ---          │
│ i64      ┆ i64         ┆ date       ┆ str       ┆ f64          │
╞══════════╪═════════════╪════════════╪═══════════╪══════════════╡
│ 1001     ┆ 1           ┆ 2023-01-10 ┆ completed ┆ 1349.98      │
│ 1002     ┆ 2           ┆ 2023-01-15 ┆ completed ┆ 599.99       │
│ 1003     ┆ 1           ┆ 2023-02-20 ┆ completed ┆ 539.97       │
│ 1004     ┆ 3           ┆ 2023-03-05 ┆ completed ┆ 1749.98      │
│ 1005     ┆ 4           ┆ 2023-03-18 ┆ cancelled ┆ 89.99        │
│ 1006     ┆ 5           ┆ 2023-04-02 ┆ completed ┆ 329.97       │
│ 1007     ┆ 2           ┆ 2023-04-22 ┆ completed ┆ 449.99       │
│ 1008     ┆ 6           ┆ 2023-05-10 ┆ completed ┆ 1299.99      │
│ …        ┆ …           ┆ …          ┆ …         ┆ …            │
│ 1019     ┆ 7           ┆ 2023-

In [45]:
order_items = pl.DataFrame({
    "item_id": list(range(1, 48)),
    "order_id": [
        1001,
        1001,
        1002,
        1003,
        1003,
        1003,
        1004,
        1004,
        1005,
        1006,
        1006,
        1006,
        1007,
        1008,
        1009,
        1010,
        1010,
        1010,
        1010,
        1011,
        1012,
        1012,
        1013,
        1013,
        1013,
        1014,
        1014,
        1015,
        1016,
        1017,
        1017,
        1018,
        1018,
        1019,
        1019,
        1020,
        1020,
        1020,
        1021,
        1021,
        1022,
        1022,
        1022,
        1023,
        1023,
        1024,
        1025,
    ],
    "product_id": [
        101,
        103,
        104,
        106,
        107,
        103,
        101,
        105,
        107,
        110,
        102,
        103,
        105,
        101,
        110,
        106,
        107,
        109,
        115,
        106,
        108,
        112,
        101,
        104,
        103,
        113,
        114,
        104,
        101,
        110,
        115,
        106,
        107,
        102,
        107,
        101,
        106,
        106,
        110,
        107,
        104,
        108,
        115,
        101,
        103,
        105,
        107,
    ],
    "quantity": [
        1,
        1,
        1,
        1,
        1,
        1,
        1,
        1,
        1,
        1,
        1,
        1,
        1,
        1,
        1,
        1,
        1,
        1,
        2,
        1,
        2,
        1,
        1,
        1,
        1,
        3,
        5,
        1,
        1,
        1,
        1,
        1,
        1,
        2,
        1,
        1,
        1,
        1,
        1,
        1,
        1,
        1,
        3,
        1,
        1,
        1,
        1,
    ],
    "unit_price": [
        1299.99,
        49.99,
        599.99,
        399.99,
        89.99,
        49.99,
        1299.99,
        449.99,
        89.99,
        249.99,
        29.99,
        49.99,
        449.99,
        1299.99,
        249.99,
        399.99,
        89.99,
        69.99,
        19.99,
        399.99,
        39.99,
        79.99,
        1299.99,
        599.99,
        49.99,
        12.99,
        8.99,
        599.99,
        1299.99,
        249.99,
        19.99,
        399.99,
        89.99,
        29.99,
        89.99,
        1299.99,
        399.99,
        399.99,
        249.99,
        89.99,
        599.99,
        39.99,
        19.99,
        1299.99,
        49.99,
        449.99,
        89.99,
    ],
})

print("=== ORDER ITEMS (first 15 rows) ===")
print(order_items.head(15))
print(f"\nTotal order items: {order_items.height}")

=== ORDER ITEMS (first 15 rows) ===
shape: (15, 5)
┌─────────┬──────────┬────────────┬──────────┬────────────┐
│ item_id ┆ order_id ┆ product_id ┆ quantity ┆ unit_price │
│ ---     ┆ ---      ┆ ---        ┆ ---      ┆ ---        │
│ i64     ┆ i64      ┆ i64        ┆ i64      ┆ f64        │
╞═════════╪══════════╪════════════╪══════════╪════════════╡
│ 1       ┆ 1001     ┆ 101        ┆ 1        ┆ 1299.99    │
│ 2       ┆ 1001     ┆ 103        ┆ 1        ┆ 49.99      │
│ 3       ┆ 1002     ┆ 104        ┆ 1        ┆ 599.99     │
│ 4       ┆ 1003     ┆ 106        ┆ 1        ┆ 399.99     │
│ 5       ┆ 1003     ┆ 107        ┆ 1        ┆ 89.99      │
│ 6       ┆ 1003     ┆ 103        ┆ 1        ┆ 49.99      │
│ 7       ┆ 1004     ┆ 101        ┆ 1        ┆ 1299.99    │
│ 8       ┆ 1004     ┆ 105        ┆ 1        ┆ 449.99     │
│ 9       ┆ 1005     ┆ 107        ┆ 1        ┆ 89.99      │
│ 10      ┆ 1006     ┆ 110        ┆ 1        ┆ 249.99     │
│ 11      ┆ 1006     ┆ 102        ┆ 1        ┆ 29

---
## 3. Schema & Data Types

Polars is strictly typed. Every column has a fixed dtype determined at creation.

In [46]:
# Schema inspection
print("=== Customers Schema ===")
print(customers.schema)

print("\n=== Column names ===")
print(customers.columns)

print("\n=== Shape (rows, cols) ===")
print(customers.shape)

print("\n=== Dtypes ===")
print(customers.dtypes)

=== Customers Schema ===
Schema({'customer_id': Int64, 'name': String, 'email': String, 'city': String, 'state': String, 'signup_date': Date})

=== Column names ===
['customer_id', 'name', 'email', 'city', 'state', 'signup_date']

=== Shape (rows, cols) ===
(10, 6)

=== Dtypes ===
[Int64, String, String, String, String, Date]


In [47]:
# Describe — summary statistics
print("=== Products describe ===")
print(products.describe())

=== Products describe ===
shape: (9, 6)
┌────────────┬────────────┬────────────────┬─────────────────┬────────────┬────────────────┐
│ statistic  ┆ product_id ┆ product_name   ┆ category        ┆ price      ┆ stock_quantity │
│ ---        ┆ ---        ┆ ---            ┆ ---             ┆ ---        ┆ ---            │
│ str        ┆ f64        ┆ str            ┆ str             ┆ f64        ┆ f64            │
╞════════════╪════════════╪════════════════╪═════════════════╪════════════╪════════════════╡
│ count      ┆ 15.0       ┆ 15             ┆ 15              ┆ 15.0       ┆ 15.0           │
│ null_count ┆ 0.0        ┆ 0              ┆ 0               ┆ 0.0        ┆ 0.0            │
│ mean       ┆ 108.0      ┆ null           ┆ null            ┆ 235.456667 ┆ 183.333333     │
│ std        ┆ 4.472136   ┆ null           ┆ null            ┆ 346.954272 ┆ 211.353349     │
│ min        ┆ 101.0      ┆ Bookshelf      ┆ Electronics     ┆ 8.99       ┆ 30.0           │
│ 25%        ┆ 105.0      ┆ nu

In [48]:
# Casting types
print("=== Casting ===")
print(
    products.select(
        pl.col("product_id").cast(pl.Utf8).alias("id_as_string"),
        pl.col("price").cast(pl.Int32).alias("price_int"),
        pl.col("stock_quantity").cast(pl.Float64).alias("stock_float"),
    )
)

=== Casting ===
shape: (15, 3)
┌──────────────┬───────────┬─────────────┐
│ id_as_string ┆ price_int ┆ stock_float │
│ ---          ┆ ---       ┆ ---         │
│ str          ┆ i32       ┆ f64         │
╞══════════════╪═══════════╪═════════════╡
│ 101          ┆ 1299      ┆ 50.0        │
│ 102          ┆ 29        ┆ 200.0       │
│ 103          ┆ 49        ┆ 150.0       │
│ 104          ┆ 599       ┆ 30.0        │
│ 105          ┆ 449       ┆ 45.0        │
│ 106          ┆ 399       ┆ 75.0        │
│ 107          ┆ 89        ┆ 120.0       │
│ 108          ┆ 39        ┆ 200.0       │
│ 109          ┆ 69        ┆ 100.0       │
│ 110          ┆ 249       ┆ 80.0        │
│ 111          ┆ 129       ┆ 60.0        │
│ 112          ┆ 79        ┆ 40.0        │
│ 113          ┆ 12        ┆ 500.0       │
│ 114          ┆ 8         ┆ 800.0       │
│ 115          ┆ 19        ┆ 300.0       │
└──────────────┴───────────┴─────────────┘


---
## 4. Basic DataFrame Operations

### Select, Filter, Sort, Slice

In [49]:
# SELECT — choose columns
print("=== Select specific columns ===")
print(customers.select("name", "city", "state").head(5))

# Select with expressions
print("\n=== Select with expressions ===")
print(
    products.select(
        pl.col("product_name"),
        pl.col("price"),
        (pl.col("price") * 0.9).round(2).alias("discounted_price"),
        pl.col("category"),
    ).head(5)
)

=== Select specific columns ===
shape: (5, 3)
┌───────────────┬─────────────┬───────┐
│ name          ┆ city        ┆ state │
│ ---           ┆ ---         ┆ ---   │
│ str           ┆ str         ┆ str   │
╞═══════════════╪═════════════╪═══════╡
│ Alice Johnson ┆ New York    ┆ NY    │
│ Bob Smith     ┆ Los Angeles ┆ CA    │
│ Charlie Brown ┆ Chicago     ┆ IL    │
│ Diana Ross    ┆ Houston     ┆ TX    │
│ Eve Wilson    ┆ Phoenix     ┆ AZ    │
└───────────────┴─────────────┴───────┘

=== Select with expressions ===
shape: (5, 4)
┌─────────────────┬─────────┬──────────────────┬─────────────┐
│ product_name    ┆ price   ┆ discounted_price ┆ category    │
│ ---             ┆ ---     ┆ ---              ┆ ---         │
│ str             ┆ f64     ┆ f64              ┆ str         │
╞═════════════════╪═════════╪══════════════════╪═════════════╡
│ Laptop Pro 15   ┆ 1299.99 ┆ 1169.99          ┆ Electronics │
│ Wireless Mouse  ┆ 29.99   ┆ 26.99            ┆ Electronics │
│ USB-C Hub       ┆ 49.99 

In [50]:
# FILTER — filter rows
print("=== Electronics products over $100 ===")
print(products.filter((pl.col("category") == "Electronics") & (pl.col("price") > 100)))

print("\n=== Customers in California ===")
print(customers.filter(pl.col("state") == "CA"))

=== Electronics products over $100 ===
shape: (3, 5)
┌────────────┬─────────────────────────────┬─────────────┬─────────┬────────────────┐
│ product_id ┆ product_name                ┆ category    ┆ price   ┆ stock_quantity │
│ ---        ┆ ---                         ┆ ---         ┆ ---     ┆ ---            │
│ i64        ┆ str                         ┆ str         ┆ f64     ┆ i64            │
╞════════════╪═════════════════════════════╪═════════════╪═════════╪════════════════╡
│ 101        ┆ Laptop Pro 15               ┆ Electronics ┆ 1299.99 ┆ 50             │
│ 106        ┆ Monitor 27"                 ┆ Electronics ┆ 399.99  ┆ 75             │
│ 110        ┆ Noise Cancelling Headphones ┆ Electronics ┆ 249.99  ┆ 80             │
└────────────┴─────────────────────────────┴─────────────┴─────────┴────────────────┘

=== Customers in California ===
shape: (2, 6)
┌─────────────┬───────────┬─────────────────┬───────────────┬───────┬─────────────┐
│ customer_id ┆ name      ┆ email         

In [51]:
# Multiple filter conditions
print("=== Completed orders over $500 ===")
print(
    orders.filter(
        (pl.col("status") == "completed")
        & (pl.col("total_amount") > 500)
        & (pl.col("order_date") >= date(2023, 6, 1))
    ).sort("total_amount", descending=True)
)

# Using is_in for multiple values
print("\n=== Customers in NY or CA ===")
print(customers.filter(pl.col("state").is_in(["NY", "CA"])))

=== Completed orders over $500 ===
shape: (6, 5)
┌──────────┬─────────────┬────────────┬───────────┬──────────────┐
│ order_id ┆ customer_id ┆ order_date ┆ status    ┆ total_amount │
│ ---      ┆ ---         ┆ ---        ┆ ---       ┆ ---          │
│ i64      ┆ i64         ┆ date       ┆ str       ┆ f64          │
╞══════════╪═════════════╪════════════╪═══════════╪══════════════╡
│ 1020     ┆ 3           ┆ 2023-11-25 ┆ completed ┆ 2099.97      │
│ 1013     ┆ 4           ┆ 2023-08-05 ┆ completed ┆ 1949.97      │
│ 1023     ┆ 9           ┆ 2023-12-28 ┆ completed ┆ 1349.98      │
│ 1010     ┆ 7           ┆ 2023-06-15 ┆ completed ┆ 869.97       │
│ 1022     ┆ 4           ┆ 2023-12-18 ┆ completed ┆ 699.98       │
│ 1015     ┆ 5           ┆ 2023-09-10 ┆ completed ┆ 599.99       │
└──────────┴─────────────┴────────────┴───────────┴──────────────┘

=== Customers in NY or CA ===
shape: (4, 6)
┌─────────────┬───────────────┬─────────────────┬───────────────┬───────┬─────────────┐
│ customer_id 

In [52]:
# SORT
print("=== Products sorted by price (descending) ===")
print(products.sort("price", descending=True).head(5))

# Multi-column sort
print("\n=== Products sorted by category (asc), then price (desc) ===")
print(products.sort(["category", "price"], descending=[False, True]))

=== Products sorted by price (descending) ===
shape: (5, 5)
┌────────────┬─────────────────────────────┬─────────────┬─────────┬────────────────┐
│ product_id ┆ product_name                ┆ category    ┆ price   ┆ stock_quantity │
│ ---        ┆ ---                         ┆ ---         ┆ ---     ┆ ---            │
│ i64        ┆ str                         ┆ str         ┆ f64     ┆ i64            │
╞════════════╪═════════════════════════════╪═════════════╪═════════╪════════════════╡
│ 101        ┆ Laptop Pro 15               ┆ Electronics ┆ 1299.99 ┆ 50             │
│ 104        ┆ Standing Desk               ┆ Furniture   ┆ 599.99  ┆ 30             │
│ 105        ┆ Ergonomic Chair             ┆ Furniture   ┆ 449.99  ┆ 45             │
│ 106        ┆ Monitor 27"                 ┆ Electronics ┆ 399.99  ┆ 75             │
│ 110        ┆ Noise Cancelling Headphones ┆ Electronics ┆ 249.99  ┆ 80             │
└────────────┴─────────────────────────────┴─────────────┴─────────┴────────────

In [53]:
# UNIQUE & N_UNIQUE
print("=== Distinct states ===")
print(customers.select("state").unique())

print(f"\nNumber of unique categories: {products['category'].n_unique()}")

# SLICE — positional row access
print("\n=== Rows 2-4 (0-indexed) ===")
print(orders.slice(2, 3))

=== Distinct states ===
shape: (7, 1)
┌───────┐
│ state │
│ ---   │
│ str   │
╞═══════╡
│ IL    │
│ NY    │
│ TX    │
│ CA    │
│ WA    │
│ AZ    │
│ MA    │
└───────┘

Number of unique categories: 3

=== Rows 2-4 (0-indexed) ===
shape: (3, 5)
┌──────────┬─────────────┬────────────┬───────────┬──────────────┐
│ order_id ┆ customer_id ┆ order_date ┆ status    ┆ total_amount │
│ ---      ┆ ---         ┆ ---        ┆ ---       ┆ ---          │
│ i64      ┆ i64         ┆ date       ┆ str       ┆ f64          │
╞══════════╪═════════════╪════════════╪═══════════╪══════════════╡
│ 1003     ┆ 1           ┆ 2023-02-20 ┆ completed ┆ 539.97       │
│ 1004     ┆ 3           ┆ 2023-03-05 ┆ completed ┆ 1749.98      │
│ 1005     ┆ 4           ┆ 2023-03-18 ┆ cancelled ┆ 89.99        │
└──────────┴─────────────┴────────────┴───────────┴──────────────┘


---
## 5. Expressions & Column Operations

Polars expressions are the heart of the API. They're composable, lazy-friendly, and parallelized.

In [54]:
# with_columns — add or modify columns
print("=== with_columns ===")
print(
    products.with_columns(
        (pl.col("price") * 1.08).round(2).alias("price_with_tax"),
        (pl.col("price") > 200).alias("is_expensive"),
        (pl.col("price") * pl.col("stock_quantity")).round(2).alias("stock_value"),
    )
)

=== with_columns ===
shape: (15, 8)
┌────────────┬────────────┬────────────┬─────────┬────────────┬────────────┬───────────┬───────────┐
│ product_id ┆ product_na ┆ category   ┆ price   ┆ stock_quan ┆ price_with ┆ is_expens ┆ stock_val │
│ ---        ┆ me         ┆ ---        ┆ ---     ┆ tity       ┆ _tax       ┆ ive       ┆ ue        │
│ i64        ┆ ---        ┆ str        ┆ f64     ┆ ---        ┆ ---        ┆ ---       ┆ ---       │
│            ┆ str        ┆            ┆         ┆ i64        ┆ f64        ┆ bool      ┆ f64       │
╞════════════╪════════════╪════════════╪═════════╪════════════╪════════════╪═══════════╪═══════════╡
│ 101        ┆ Laptop Pro ┆ Electronic ┆ 1299.99 ┆ 50         ┆ 1403.99    ┆ true      ┆ 64999.5   │
│            ┆ 15         ┆ s          ┆         ┆            ┆            ┆           ┆           │
│ 102        ┆ Wireless   ┆ Electronic ┆ 29.99   ┆ 200        ┆ 32.39      ┆ false     ┆ 5998.0    │
│            ┆ Mouse      ┆ s          ┆         ┆     

In [55]:
# WHEN / THEN / OTHERWISE — conditional expressions
print("=== Price Tiers ===")
print(
    products.with_columns(
        pl
        .when(pl.col("price") < 50)
        .then(pl.lit("Budget"))
        .when(pl.col("price") < 200)
        .then(pl.lit("Mid-Range"))
        .when(pl.col("price") < 500)
        .then(pl.lit("Premium"))
        .otherwise(pl.lit("Luxury"))
        .alias("price_tier")
    ).select("product_name", "price", "price_tier")
)

=== Price Tiers ===
shape: (15, 3)
┌─────────────────────────────┬─────────┬────────────┐
│ product_name                ┆ price   ┆ price_tier │
│ ---                         ┆ ---     ┆ ---        │
│ str                         ┆ f64     ┆ str        │
╞═════════════════════════════╪═════════╪════════════╡
│ Laptop Pro 15               ┆ 1299.99 ┆ Luxury     │
│ Wireless Mouse              ┆ 29.99   ┆ Budget     │
│ USB-C Hub                   ┆ 49.99   ┆ Budget     │
│ Standing Desk               ┆ 599.99  ┆ Luxury     │
│ Ergonomic Chair             ┆ 449.99  ┆ Premium    │
│ Monitor 27"                 ┆ 399.99  ┆ Premium    │
│ Mechanical Keyboard         ┆ 89.99   ┆ Mid-Range  │
│ Desk Lamp                   ┆ 39.99   ┆ Budget     │
│ Webcam HD                   ┆ 69.99   ┆ Mid-Range  │
│ Noise Cancelling Headphones ┆ 249.99  ┆ Premium    │
│ Bookshelf                   ┆ 129.99  ┆ Mid-Range  │
│ Whiteboard                  ┆ 79.99   ┆ Mid-Range  │
│ Notebook Set                

In [56]:
# Multiple expressions in one select (parallel execution)
print("=== Multiple expressions ===")
print(
    products.select(
        pl.col("product_name"),
        pl.col("price"),
        pl.col("price").mean().alias("avg_price"),
        (pl.col("price") - pl.col("price").mean()).round(2).alias("price_vs_avg"),
        (pl.col("price") / pl.col("price").sum() * 100).round(2).alias("pct_of_total"),
    )
)

=== Multiple expressions ===
shape: (15, 5)
┌─────────────────────────────┬─────────┬────────────┬──────────────┬──────────────┐
│ product_name                ┆ price   ┆ avg_price  ┆ price_vs_avg ┆ pct_of_total │
│ ---                         ┆ ---     ┆ ---        ┆ ---          ┆ ---          │
│ str                         ┆ f64     ┆ f64        ┆ f64          ┆ f64          │
╞═════════════════════════════╪═════════╪════════════╪══════════════╪══════════════╡
│ Laptop Pro 15               ┆ 1299.99 ┆ 235.456667 ┆ 1064.53      ┆ 36.81        │
│ Wireless Mouse              ┆ 29.99   ┆ 235.456667 ┆ -205.47      ┆ 0.85         │
│ USB-C Hub                   ┆ 49.99   ┆ 235.456667 ┆ -185.47      ┆ 1.42         │
│ Standing Desk               ┆ 599.99  ┆ 235.456667 ┆ 364.53       ┆ 16.99        │
│ Ergonomic Chair             ┆ 449.99  ┆ 235.456667 ┆ 214.53       ┆ 12.74        │
│ Monitor 27"                 ┆ 399.99  ┆ 235.456667 ┆ 164.53       ┆ 11.33        │
│ Mechanical Keyboard

In [57]:
# Column selectors — select by dtype or pattern
import polars.selectors as cs

print("=== Numeric columns only ===")
print(products.select(cs.numeric()))

print("\n=== String columns only ===")
print(customers.select(cs.string()).head(5))

=== Numeric columns only ===
shape: (15, 3)
┌────────────┬─────────┬────────────────┐
│ product_id ┆ price   ┆ stock_quantity │
│ ---        ┆ ---     ┆ ---            │
│ i64        ┆ f64     ┆ i64            │
╞════════════╪═════════╪════════════════╡
│ 101        ┆ 1299.99 ┆ 50             │
│ 102        ┆ 29.99   ┆ 200            │
│ 103        ┆ 49.99   ┆ 150            │
│ 104        ┆ 599.99  ┆ 30             │
│ 105        ┆ 449.99  ┆ 45             │
│ 106        ┆ 399.99  ┆ 75             │
│ 107        ┆ 89.99   ┆ 120            │
│ 108        ┆ 39.99   ┆ 200            │
│ 109        ┆ 69.99   ┆ 100            │
│ 110        ┆ 249.99  ┆ 80             │
│ 111        ┆ 129.99  ┆ 60             │
│ 112        ┆ 79.99   ┆ 40             │
│ 113        ┆ 12.99   ┆ 500            │
│ 114        ┆ 8.99    ┆ 800            │
│ 115        ┆ 19.99   ┆ 300            │
└────────────┴─────────┴────────────────┘

=== String columns only ===
shape: (5, 4)
┌───────────────┬──────────────

---
## 6. Aggregations

### group_by, agg, multiple aggregations

In [58]:
# Basic group_by
print("=== Order Statistics by Status ===")
print(
    orders
    .group_by("status")
    .agg(
        pl.len().alias("order_count"),
        pl.col("total_amount").sum().round(2).alias("total_revenue"),
        pl.col("total_amount").mean().round(2).alias("avg_order_value"),
        pl.col("total_amount").min().alias("min_order"),
        pl.col("total_amount").max().alias("max_order"),
    )
    .sort("order_count", descending=True)
)

=== Order Statistics by Status ===
shape: (4, 6)
┌───────────┬─────────────┬───────────────┬─────────────────┬───────────┬───────────┐
│ status    ┆ order_count ┆ total_revenue ┆ avg_order_value ┆ min_order ┆ max_order │
│ ---       ┆ ---         ┆ ---           ┆ ---             ┆ ---       ┆ ---       │
│ str       ┆ u32         ┆ f64           ┆ f64             ┆ f64       ┆ f64       │
╞═══════════╪═════════════╪═══════════════╪═════════════════╪═══════════╪═══════════╡
│ completed ┆ 21          ┆ 16229.59      ┆ 772.84          ┆ 79.98     ┆ 2099.97   │
│ cancelled ┆ 2           ┆ 1389.98       ┆ 694.99          ┆ 89.99     ┆ 1299.99   │
│ pending   ┆ 1           ┆ 89.99         ┆ 89.99           ┆ 89.99     ┆ 89.99     │
│ returned  ┆ 1           ┆ 249.99        ┆ 249.99          ┆ 249.99    ┆ 249.99    │
└───────────┴─────────────┴───────────────┴─────────────────┴───────────┴───────────┘


In [59]:
# Multi-table aggregation
print("=== Product Sales by Category ===")
print(
    order_items
    .join(products, on="product_id")
    .group_by("category")
    .agg(
        pl.len().alias("items_sold"),
        (pl.col("quantity") * pl.col("unit_price"))
        .sum()
        .round(2)
        .alias("total_revenue"),
        pl.col("product_id").n_unique().alias("unique_products"),
        pl.col("unit_price").mean().round(2).alias("avg_price"),
    )
    .sort("total_revenue", descending=True)
)

=== Product Sales by Category ===
shape: (3, 5)
┌─────────────────┬────────────┬───────────────┬─────────────────┬───────────┐
│ category        ┆ items_sold ┆ total_revenue ┆ unique_products ┆ avg_price │
│ ---             ┆ ---        ┆ ---           ┆ ---             ┆ ---       │
│ str             ┆ u32        ┆ f64           ┆ u32             ┆ f64       │
╞═════════════════╪════════════╪═══════════════╪═════════════════╪═══════════╡
│ Electronics     ┆ 32         ┆ 13539.67      ┆ 7               ┆ 422.18    │
│ Furniture       ┆ 9          ┆ 3869.9        ┆ 3               ┆ 425.55    │
│ Office Supplies ┆ 6          ┆ 283.85        ┆ 4               ┆ 26.99     │
└─────────────────┴────────────┴───────────────┴─────────────────┴───────────┘


In [60]:
# Monthly revenue trend
print("=== Monthly Revenue (2023) ===")
print(
    orders
    .filter(
        (pl.col("status") == "completed") & (pl.col("order_date").dt.year() == 2023)
    )
    .with_columns(pl.col("order_date").dt.month().alias("month"))
    .group_by("month")
    .agg(
        pl.len().alias("orders"),
        pl.col("total_amount").sum().round(2).alias("revenue"),
    )
    .sort("month")
)

=== Monthly Revenue (2023) ===
shape: (12, 3)
┌───────┬────────┬─────────┐
│ month ┆ orders ┆ revenue │
│ ---   ┆ ---    ┆ ---     │
│ i8    ┆ u32    ┆ f64     │
╞═══════╪════════╪═════════╡
│ 1     ┆ 2      ┆ 1949.97 │
│ 2     ┆ 1      ┆ 539.97  │
│ 3     ┆ 1      ┆ 1749.98 │
│ 4     ┆ 2      ┆ 779.96  │
│ 5     ┆ 1      ┆ 1299.99 │
│ 6     ┆ 1      ┆ 869.97  │
│ 7     ┆ 2      ┆ 559.97  │
│ 8     ┆ 2      ┆ 2029.95 │
│ 9     ┆ 1      ┆ 599.99  │
│ 10    ┆ 2      ┆ 759.96  │
│ 11    ┆ 2      ┆ 2249.95 │
│ 12    ┆ 3      ┆ 2389.94 │
└───────┴────────┴─────────┘


In [61]:
# Collecting values into lists
print("=== Products per Category (as list) ===")
print(
    products.group_by("category").agg(
        pl.col("product_name").alias("products"),
        pl.col("product_name").n_unique().alias("count"),
    )
)

=== Products per Category (as list) ===
shape: (3, 3)
┌─────────────────┬─────────────────────────────────────────────────────┬───────┐
│ category        ┆ products                                            ┆ count │
│ ---             ┆ ---                                                 ┆ ---   │
│ str             ┆ list[str]                                           ┆ u32   │
╞═════════════════╪═════════════════════════════════════════════════════╪═══════╡
│ Furniture       ┆ ["Standing Desk", "Ergonomic Chair", … "Bookshelf"… ┆ 4     │
│ Office Supplies ┆ ["Whiteboard", "Notebook Set", … "Cable Management… ┆ 4     │
│ Electronics     ┆ ["Laptop Pro 15", "Wireless Mouse", … "Noise Cance… ┆ 7     │
└─────────────────┴─────────────────────────────────────────────────────┴───────┘


---
## 7. Joins

Polars supports: `inner`, `left`, `right`, `full`, `cross`, `semi`, `anti`

In [62]:
# INNER JOIN
print("=== Inner Join: Orders with Customer Names ===")
print(
    orders
    .join(customers, on="customer_id", how="inner")
    .select("order_id", "name", "order_date", "total_amount", "status")
    .sort("order_date")
    .head(10)
)

=== Inner Join: Orders with Customer Names ===
shape: (10, 5)
┌──────────┬───────────────┬────────────┬──────────────┬───────────┐
│ order_id ┆ name          ┆ order_date ┆ total_amount ┆ status    │
│ ---      ┆ ---           ┆ ---        ┆ ---          ┆ ---       │
│ i64      ┆ str           ┆ date       ┆ f64          ┆ str       │
╞══════════╪═══════════════╪════════════╪══════════════╪═══════════╡
│ 1001     ┆ Alice Johnson ┆ 2023-01-10 ┆ 1349.98      ┆ completed │
│ 1002     ┆ Bob Smith     ┆ 2023-01-15 ┆ 599.99       ┆ completed │
│ 1003     ┆ Alice Johnson ┆ 2023-02-20 ┆ 539.97       ┆ completed │
│ 1004     ┆ Charlie Brown ┆ 2023-03-05 ┆ 1749.98      ┆ completed │
│ 1005     ┆ Diana Ross    ┆ 2023-03-18 ┆ 89.99        ┆ cancelled │
│ 1006     ┆ Eve Wilson    ┆ 2023-04-02 ┆ 329.97       ┆ completed │
│ 1007     ┆ Bob Smith     ┆ 2023-04-22 ┆ 449.99       ┆ completed │
│ 1008     ┆ Frank Miller  ┆ 2023-05-10 ┆ 1299.99      ┆ completed │
│ 1009     ┆ Charlie Brown ┆ 2023-05-28 ┆

In [63]:
# LEFT JOIN with aggregation
print("=== Left Join: All Customers with Order Count ===")
order_counts = orders.group_by("customer_id").agg(pl.len().alias("order_count"))
print(
    customers
    .join(order_counts, on="customer_id", how="left")
    .with_columns(pl.col("order_count").fill_null(0))
    .select("name", "city", "order_count")
    .sort("order_count", descending=True)
)

=== Left Join: All Customers with Order Count ===
shape: (10, 3)
┌───────────────┬───────────────┬─────────────┐
│ name          ┆ city          ┆ order_count │
│ ---           ┆ ---           ┆ ---         │
│ str           ┆ str           ┆ u32         │
╞═══════════════╪═══════════════╪═════════════╡
│ Alice Johnson ┆ New York      ┆ 4           │
│ Bob Smith     ┆ Los Angeles   ┆ 3           │
│ Charlie Brown ┆ Chicago       ┆ 3           │
│ Diana Ross    ┆ Houston       ┆ 3           │
│ Eve Wilson    ┆ Phoenix       ┆ 3           │
│ Frank Miller  ┆ New York      ┆ 2           │
│ Grace Lee     ┆ San Francisco ┆ 2           │
│ Henry Davis   ┆ Chicago       ┆ 2           │
│ Ivy Chen      ┆ Seattle       ┆ 2           │
│ Jack Thompson ┆ Boston        ┆ 1           │
└───────────────┴───────────────┴─────────────┘


In [64]:
# ANTI JOIN — rows in left with NO match in right
print("=== Anti Join: Customers with NO orders ===")
print(
    customers.join(orders, on="customer_id", how="anti").select(
        "customer_id", "name", "city"
    )
)

# SEMI JOIN — rows in left that DO have a match (like EXISTS)
print("\n=== Semi Join: Customers WITH orders ===")
print(
    customers.join(orders, on="customer_id", how="semi").select(
        "customer_id", "name", "city"
    )
)

=== Anti Join: Customers with NO orders ===
shape: (0, 3)
┌─────────────┬──────┬──────┐
│ customer_id ┆ name ┆ city │
│ ---         ┆ ---  ┆ ---  │
│ i64         ┆ str  ┆ str  │
╞═════════════╪══════╪══════╡
└─────────────┴──────┴──────┘

=== Semi Join: Customers WITH orders ===
shape: (10, 3)
┌─────────────┬───────────────┬───────────────┐
│ customer_id ┆ name          ┆ city          │
│ ---         ┆ ---           ┆ ---           │
│ i64         ┆ str           ┆ str           │
╞═════════════╪═══════════════╪═══════════════╡
│ 1           ┆ Alice Johnson ┆ New York      │
│ 2           ┆ Bob Smith     ┆ Los Angeles   │
│ 3           ┆ Charlie Brown ┆ Chicago       │
│ 4           ┆ Diana Ross    ┆ Houston       │
│ 5           ┆ Eve Wilson    ┆ Phoenix       │
│ 6           ┆ Frank Miller  ┆ New York      │
│ 7           ┆ Grace Lee     ┆ San Francisco │
│ 8           ┆ Henry Davis   ┆ Chicago       │
│ 9           ┆ Ivy Chen      ┆ Seattle       │
│ 10          ┆ Jack Thompson ┆ B

In [65]:
# MULTI-TABLE JOIN
print("=== Complete Order Details ===")
full_details = (
    order_items
    .join(orders, on="order_id")
    .join(customers, on="customer_id")
    .join(products, on="product_id")
    .select(
        "order_id",
        "name",
        "order_date",
        "product_name",
        "category",
        "quantity",
        "unit_price",
        (pl.col("quantity") * pl.col("unit_price")).round(2).alias("line_total"),
    )
    .sort("order_id", "product_name")
)
print(full_details.head(15))

=== Complete Order Details ===
shape: (15, 8)
┌──────────┬────────────┬────────────┬────────────┬────────────┬──────────┬────────────┬───────────┐
│ order_id ┆ name       ┆ order_date ┆ product_na ┆ category   ┆ quantity ┆ unit_price ┆ line_tota │
│ ---      ┆ ---        ┆ ---        ┆ me         ┆ ---        ┆ ---      ┆ ---        ┆ l         │
│ i64      ┆ str        ┆ date       ┆ ---        ┆ str        ┆ i64      ┆ f64        ┆ ---       │
│          ┆            ┆            ┆ str        ┆            ┆          ┆            ┆ f64       │
╞══════════╪════════════╪════════════╪════════════╪════════════╪══════════╪════════════╪═══════════╡
│ 1001     ┆ Alice      ┆ 2023-01-10 ┆ Laptop Pro ┆ Electronic ┆ 1        ┆ 1299.99    ┆ 1299.99   │
│          ┆ Johnson    ┆            ┆ 15         ┆ s          ┆          ┆            ┆           │
│ 1001     ┆ Alice      ┆ 2023-01-10 ┆ USB-C Hub  ┆ Electronic ┆ 1        ┆ 49.99      ┆ 49.99     │
│          ┆ Johnson    ┆            ┆       

In [66]:
# CROSS JOIN
print("=== Cross Join: All Category-State Combinations ===")
categories = products.select("category").unique()
states = customers.select("state").unique()
print(categories.join(states, how="cross").sort("category", "state").head(10))

=== Cross Join: All Category-State Combinations ===
shape: (10, 2)
┌─────────────┬───────┐
│ category    ┆ state │
│ ---         ┆ ---   │
│ str         ┆ str   │
╞═════════════╪═══════╡
│ Electronics ┆ AZ    │
│ Electronics ┆ CA    │
│ Electronics ┆ IL    │
│ Electronics ┆ MA    │
│ Electronics ┆ NY    │
│ Electronics ┆ TX    │
│ Electronics ┆ WA    │
│ Furniture   ┆ AZ    │
│ Furniture   ┆ CA    │
│ Furniture   ┆ IL    │
└─────────────┴───────┘


---
## 8. Window Functions

Polars uses `.over()` for window operations — computations partitioned by group without collapsing rows.

Key pattern: `expression.over("partition_col")`

In [67]:
# RANKING within groups
print("=== Product Rankings by Category ===")
print(
    products
    .with_columns(
        pl
        .col("price")
        .rank(method="ordinal", descending=True)
        .over("category")
        .alias("row_num"),
        pl
        .col("price")
        .rank(method="min", descending=True)
        .over("category")
        .alias("rank"),
        pl
        .col("price")
        .rank(method="dense", descending=True)
        .over("category")
        .alias("dense_rank"),
    )
    .select("category", "product_name", "price", "row_num", "rank", "dense_rank")
    .sort("category", "row_num")
)

=== Product Rankings by Category ===
shape: (15, 6)
┌─────────────────┬─────────────────────────────┬─────────┬─────────┬──────┬────────────┐
│ category        ┆ product_name                ┆ price   ┆ row_num ┆ rank ┆ dense_rank │
│ ---             ┆ ---                         ┆ ---     ┆ ---     ┆ ---  ┆ ---        │
│ str             ┆ str                         ┆ f64     ┆ u32     ┆ u32  ┆ u32        │
╞═════════════════╪═════════════════════════════╪═════════╪═════════╪══════╪════════════╡
│ Electronics     ┆ Laptop Pro 15               ┆ 1299.99 ┆ 1       ┆ 1    ┆ 1          │
│ Electronics     ┆ Monitor 27"                 ┆ 399.99  ┆ 2       ┆ 2    ┆ 2          │
│ Electronics     ┆ Noise Cancelling Headphones ┆ 249.99  ┆ 3       ┆ 3    ┆ 3          │
│ Electronics     ┆ Mechanical Keyboard         ┆ 89.99   ┆ 4       ┆ 4    ┆ 4          │
│ Electronics     ┆ Webcam HD                   ┆ 69.99   ┆ 5       ┆ 5    ┆ 5          │
│ Electronics     ┆ USB-C Hub                   

In [68]:
# LAG / LEAD (shift) — access previous/next row values
print("=== Customer Order History with Lag/Lead ===")
print(
    orders
    .filter(pl.col("status") == "completed")
    .sort("customer_id", "order_date")
    .with_columns(
        pl.col("total_amount").shift(1).over("customer_id").alias("prev_order_amount"),
        pl.col("total_amount").shift(-1).over("customer_id").alias("next_order_amount"),
        pl.col("order_date").shift(1).over("customer_id").alias("prev_order_date"),
    )
    .with_columns(
        (pl.col("order_date") - pl.col("prev_order_date"))
        .dt.total_days()
        .alias("days_between_orders")
    )
    .join(customers.select("customer_id", "name"), on="customer_id")
    .select(
        "name", "order_date", "total_amount", "prev_order_amount", "days_between_orders"
    )
    .head(20)
)

=== Customer Order History with Lag/Lead ===
shape: (20, 5)
┌───────────────┬────────────┬──────────────┬───────────────────┬─────────────────────┐
│ name          ┆ order_date ┆ total_amount ┆ prev_order_amount ┆ days_between_orders │
│ ---           ┆ ---        ┆ ---          ┆ ---               ┆ ---                 │
│ str           ┆ date       ┆ f64          ┆ f64               ┆ i64                 │
╞═══════════════╪════════════╪══════════════╪═══════════════════╪═════════════════════╡
│ Alice Johnson ┆ 2023-01-10 ┆ 1349.98      ┆ null              ┆ null                │
│ Alice Johnson ┆ 2023-02-20 ┆ 539.97       ┆ 1349.98           ┆ 41                  │
│ Alice Johnson ┆ 2023-07-01 ┆ 399.99       ┆ 539.97            ┆ 131                 │
│ Alice Johnson ┆ 2023-12-05 ┆ 339.98       ┆ 399.99            ┆ 157                 │
│ Bob Smith     ┆ 2023-01-15 ┆ 599.99       ┆ null              ┆ null                │
│ Bob Smith     ┆ 2023-04-22 ┆ 449.99       ┆ 599.99        

In [69]:
# CUMULATIVE aggregates
print("=== Cumulative Spending per Customer ===")
print(
    orders
    .filter(pl.col("status") == "completed")
    .sort("customer_id", "order_date")
    .with_columns(
        pl
        .col("total_amount")
        .cum_sum()
        .over("customer_id")
        .round(2)
        .alias("cumulative_spend"),
        (
            pl.col("total_amount").cum_sum().over("customer_id")
            / pl.col("total_amount").cum_count().over("customer_id")
        )
        .round(2)
        .alias("running_avg"),
        pl.col("total_amount").cum_count().over("customer_id").alias("order_num"),
    )
    .join(customers.select("customer_id", "name"), on="customer_id")
    .filter(pl.col("customer_id").is_in([1, 2, 3]))
    .select(
        "name",
        "order_num",
        "order_date",
        "total_amount",
        "cumulative_spend",
        "running_avg",
    )
    .sort("name", "order_date")
)

=== Cumulative Spending per Customer ===
shape: (9, 6)
┌───────────────┬───────────┬────────────┬──────────────┬──────────────────┬─────────────┐
│ name          ┆ order_num ┆ order_date ┆ total_amount ┆ cumulative_spend ┆ running_avg │
│ ---           ┆ ---       ┆ ---        ┆ ---          ┆ ---              ┆ ---         │
│ str           ┆ u32       ┆ date       ┆ f64          ┆ f64              ┆ f64         │
╞═══════════════╪═══════════╪════════════╪══════════════╪══════════════════╪═════════════╡
│ Alice Johnson ┆ 1         ┆ 2023-01-10 ┆ 1349.98      ┆ 1349.98          ┆ 1349.98     │
│ Alice Johnson ┆ 2         ┆ 2023-02-20 ┆ 539.97       ┆ 1889.95          ┆ 944.98      │
│ Alice Johnson ┆ 3         ┆ 2023-07-01 ┆ 399.99       ┆ 2289.94          ┆ 763.31      │
│ Alice Johnson ┆ 4         ┆ 2023-12-05 ┆ 339.98       ┆ 2629.92          ┆ 657.48      │
│ Bob Smith     ┆ 1         ┆ 2023-01-15 ┆ 599.99       ┆ 599.99           ┆ 599.99      │
│ Bob Smith     ┆ 2         ┆ 2023-

In [70]:
# Percent of total within group
print("=== Product Price as % of Category Total ===")
print(
    products
    .with_columns(
        pl.col("price").sum().over("category").alias("category_total"),
    )
    .with_columns(
        (pl.col("price") / pl.col("category_total") * 100)
        .round(1)
        .alias("pct_of_category")
    )
    .select("category", "product_name", "price", "category_total", "pct_of_category")
    .sort("category", "price", descending=[False, True])
)

=== Product Price as % of Category Total ===
shape: (15, 5)
┌─────────────────┬─────────────────────────────┬─────────┬────────────────┬─────────────────┐
│ category        ┆ product_name                ┆ price   ┆ category_total ┆ pct_of_category │
│ ---             ┆ ---                         ┆ ---     ┆ ---            ┆ ---             │
│ str             ┆ str                         ┆ f64     ┆ f64            ┆ f64             │
╞═════════════════╪═════════════════════════════╪═════════╪════════════════╪═════════════════╡
│ Electronics     ┆ Laptop Pro 15               ┆ 1299.99 ┆ 2189.93        ┆ 59.4            │
│ Electronics     ┆ Monitor 27"                 ┆ 399.99  ┆ 2189.93        ┆ 18.3            │
│ Electronics     ┆ Noise Cancelling Headphones ┆ 249.99  ┆ 2189.93        ┆ 11.4            │
│ Electronics     ┆ Mechanical Keyboard         ┆ 89.99   ┆ 2189.93        ┆ 4.1             │
│ Electronics     ┆ Webcam HD                   ┆ 69.99   ┆ 2189.93        ┆ 3.2     

---
## 9. Nested Data & Structs

Polars natively supports List and Struct dtypes — no need to flatten everything.

In [71]:
# List columns — from group_by
print("=== List columns ===")
category_products = products.group_by("category").agg(
    pl.col("product_name"),
    pl.col("price"),
)
print(category_products)

# Access list elements
print("\n=== First product per category ===")
print(
    category_products.with_columns(
        pl.col("product_name").list.first().alias("first_product"),
        pl.col("price").list.max().alias("max_price"),
        pl.col("price").list.len().alias("num_products"),
    ).select("category", "first_product", "max_price", "num_products")
)

=== List columns ===
shape: (3, 3)
┌─────────────────┬────────────────────────────────────────────────┬────────────────────────────┐
│ category        ┆ product_name                                   ┆ price                      │
│ ---             ┆ ---                                            ┆ ---                        │
│ str             ┆ list[str]                                      ┆ list[f64]                  │
╞═════════════════╪════════════════════════════════════════════════╪════════════════════════════╡
│ Electronics     ┆ ["Laptop Pro 15", "Wireless Mouse", … "Noise   ┆ [1299.99, 29.99, … 249.99] │
│                 ┆ Cance…                                         ┆                            │
│ Office Supplies ┆ ["Whiteboard", "Notebook Set", … "Cable        ┆ [79.99, 12.99, … 19.99]    │
│                 ┆ Management…                                    ┆                            │
│ Furniture       ┆ ["Standing Desk", "Ergonomic Chair", …         ┆ [599.99, 449.9

In [72]:
# Struct columns
print("=== Struct column ===")
df_struct = pl.DataFrame({
    "id": [1, 2, 3],
    "address": [
        {"street": "123 Main St", "city": "NYC", "zip": "10001"},
        {"street": "456 Oak Ave", "city": "LA", "zip": "90001"},
        {"street": "789 Pine Rd", "city": "Chicago", "zip": "60601"},
    ],
})
print(df_struct)

# Unnest struct fields
print("\n=== Unnested ===")
print(df_struct.unnest("address"))

# Access struct fields
print("\n=== Access struct field ===")
print(df_struct.select("id", pl.col("address").struct.field("city").alias("city")))

=== Struct column ===
shape: (3, 2)
┌─────┬───────────────────────────────────┐
│ id  ┆ address                           │
│ --- ┆ ---                               │
│ i64 ┆ struct[3]                         │
╞═════╪═══════════════════════════════════╡
│ 1   ┆ {"123 Main St","NYC","10001"}     │
│ 2   ┆ {"456 Oak Ave","LA","90001"}      │
│ 3   ┆ {"789 Pine Rd","Chicago","60601"} │
└─────┴───────────────────────────────────┘

=== Unnested ===
shape: (3, 4)
┌─────┬─────────────┬─────────┬───────┐
│ id  ┆ street      ┆ city    ┆ zip   │
│ --- ┆ ---         ┆ ---     ┆ ---   │
│ i64 ┆ str         ┆ str     ┆ str   │
╞═════╪═════════════╪═════════╪═══════╡
│ 1   ┆ 123 Main St ┆ NYC     ┆ 10001 │
│ 2   ┆ 456 Oak Ave ┆ LA      ┆ 90001 │
│ 3   ┆ 789 Pine Rd ┆ Chicago ┆ 60601 │
└─────┴─────────────┴─────────┴───────┘

=== Access struct field ===
shape: (3, 2)
┌─────┬─────────┐
│ id  ┆ city    │
│ --- ┆ ---     │
│ i64 ┆ str     │
╞═════╪═════════╡
│ 1   ┆ NYC     │
│ 2   ┆ LA      │
│ 3   ┆

---
## 10. Lazy Evaluation & Query Optimization

Polars' **LazyFrame** builds a logical plan and optimizes it before execution:
- Predicate pushdown
- Projection pushdown
- Common subexpression elimination
- Slice pushdown

In [73]:
# Lazy mode — build a plan, then collect
lazy_result = (
    orders
    .lazy()
    .filter(pl.col("status") == "completed")
    .join(customers.lazy(), on="customer_id")
    .group_by("state")
    .agg(
        pl.len().alias("order_count"),
        pl.col("total_amount").sum().round(2).alias("total_revenue"),
    )
    .sort("total_revenue", descending=True)
)

# Inspect the plan
print("=== Optimized Query Plan ===")
print(lazy_result.explain())

# Execute
print("\n=== Result ===")
print(lazy_result.collect())

=== Optimized Query Plan ===
SORT BY [descending: [true]] [col("total_revenue")]
  AGGREGATE[maintain_order: false]
    [len().alias("order_count"), col("total_amount").sum().round().alias("total_revenue")] BY [col("state")]
    FROM
    simple π 2/2 ["state", "total_amount"]
      INNER JOIN:
      LEFT PLAN ON: [col("customer_id")]
        simple π 2/2 ["total_amount", "customer_id"]
          FILTER col("status") == "completed"
          FROM
            DF ["order_id", "customer_id", "order_date", "status", ...]; PROJECT["total_amount", "customer_id", "status"] 3/5 COLUMNS
      RIGHT PLAN ON: [col("customer_id")]
        DF ["customer_id", "name", "email", "city", ...]; PROJECT["state", "customer_id"] 2/6 COLUMNS
      END INNER JOIN

=== Result ===
shape: (6, 3)
┌───────┬─────────────┬───────────────┐
│ state ┆ order_count ┆ total_revenue │
│ ---   ┆ ---         ┆ ---           │
│ str   ┆ u32         ┆ f64           │
╞═══════╪═════════════╪═══════════════╡
│ IL    ┆ 4          

In [74]:
# Comparing lazy vs eager — lazy is preferred for complex pipelines
# Lazy defers execution and applies optimizations

complex_query = (
    order_items
    .lazy()
    .join(orders.lazy(), on="order_id")
    .filter(pl.col("status") == "completed")
    .join(products.lazy(), on="product_id")
    .group_by("category")
    .agg(
        (pl.col("quantity") * pl.col("unit_price")).sum().round(2).alias("revenue"),
        pl.col("product_id").n_unique().alias("unique_products"),
    )
    .sort("revenue", descending=True)
)

print("=== Optimized Plan (notice predicate pushdown) ===")
print(complex_query.explain())
print("\n=== Result ===")
print(complex_query.collect())

=== Optimized Plan (notice predicate pushdown) ===
SORT BY [descending: [true]] [col("revenue")]
  AGGREGATE[maintain_order: false]
    [(col("quantity").cast(Float64) * col("unit_price")).sum().round().alias("revenue"), col("product_id").n_unique().alias("unique_products")] BY [col("category")]
    FROM
    simple π 4/4 ["category", "quantity", ... 2 other columns]
      INNER JOIN:
      LEFT PLAN ON: [col("product_id")]
        simple π 3/3 ["product_id", "quantity", ... 1 other column]
          INNER JOIN:
          LEFT PLAN ON: [col("order_id")]
            DF ["item_id", "order_id", "product_id", "quantity", ...]; PROJECT["product_id", "quantity", "unit_price", "order_id"] 4/5 COLUMNS
          RIGHT PLAN ON: [col("order_id")]
            simple π 1/1 ["order_id"]
              FILTER col("status") == "completed"
              FROM
                DF ["order_id", "customer_id", "order_date", "status", ...]; PROJECT["order_id", "status"] 2/5 COLUMNS
          END INNER JOIN
    

In [75]:
# Profile query execution
print("=== Query Profile ===")
result_df, timing_df = (
    orders
    .lazy()
    .filter(pl.col("status") == "completed")
    .group_by(pl.col("order_date").dt.month().alias("month"))
    .agg(pl.col("total_amount").sum().alias("revenue"))
    .sort("month")
    .profile()
)
print(result_df)
print("\n=== Timing ===")
print(timing_df)

=== Query Profile ===
shape: (12, 2)
┌───────┬─────────┐
│ month ┆ revenue │
│ ---   ┆ ---     │
│ i8    ┆ f64     │
╞═══════╪═════════╡
│ 1     ┆ 2399.96 │
│ 2     ┆ 539.97  │
│ 3     ┆ 1749.98 │
│ 4     ┆ 779.96  │
│ 5     ┆ 1299.99 │
│ 6     ┆ 869.97  │
│ 7     ┆ 559.97  │
│ 8     ┆ 2029.95 │
│ 9     ┆ 599.99  │
│ 10    ┆ 759.96  │
│ 11    ┆ 2249.95 │
│ 12    ┆ 2389.94 │
└───────┴─────────┘

=== Timing ===
shape: (5, 3)
┌─────────────────────────────────────────────┬───────┬─────┐
│ node                                        ┆ start ┆ end │
│ ---                                         ┆ ---   ┆ --- │
│ str                                         ┆ u64   ┆ u64 │
╞═════════════════════════════════════════════╪═══════╪═════╡
│ optimization                                ┆ 0     ┆ 54  │
│ .filter([(col("status")) == ("completed")]) ┆ 58    ┆ 211 │
│ simple-projection(order_date, total_amount) ┆ 212   ┆ 214 │
│ group_by(month)                             ┆ 217   ┆ 335 │
│ sort(month)  

---
## 11. Reading & Writing Data

Polars supports CSV, JSON, Parquet, IPC/Arrow, and more.

In [76]:
import os
import tempfile

output_dir = os.path.join(tempfile.gettempdir(), "polars_tutorial_output")
os.makedirs(output_dir, exist_ok=True)
print(f"Output directory: {output_dir}")

Output directory: /var/folders/19/_f944hvd0sd_72vpz49q73pw0000gn/T/polars_tutorial_output


In [77]:
# WRITE & READ CSV
csv_path = os.path.join(output_dir, "products.csv")
products.write_csv(csv_path)
print(f"Written CSV to: {csv_path}")

products_from_csv = pl.read_csv(csv_path)
print("\n=== Read back from CSV ===")
print(products_from_csv.head(5))

Written CSV to: /var/folders/19/_f944hvd0sd_72vpz49q73pw0000gn/T/polars_tutorial_output/products.csv

=== Read back from CSV ===
shape: (5, 5)
┌────────────┬─────────────────┬─────────────┬─────────┬────────────────┐
│ product_id ┆ product_name    ┆ category    ┆ price   ┆ stock_quantity │
│ ---        ┆ ---             ┆ ---         ┆ ---     ┆ ---            │
│ i64        ┆ str             ┆ str         ┆ f64     ┆ i64            │
╞════════════╪═════════════════╪═════════════╪═════════╪════════════════╡
│ 101        ┆ Laptop Pro 15   ┆ Electronics ┆ 1299.99 ┆ 50             │
│ 102        ┆ Wireless Mouse  ┆ Electronics ┆ 29.99   ┆ 200            │
│ 103        ┆ USB-C Hub       ┆ Electronics ┆ 49.99   ┆ 150            │
│ 104        ┆ Standing Desk   ┆ Furniture   ┆ 599.99  ┆ 30             │
│ 105        ┆ Ergonomic Chair ┆ Furniture   ┆ 449.99  ┆ 45             │
└────────────┴─────────────────┴─────────────┴─────────┴────────────────┘


In [78]:
# WRITE & READ JSON (newline-delimited)
json_path = os.path.join(output_dir, "orders.ndjson")
orders.write_ndjson(json_path)
print(f"Written NDJSON to: {json_path}")

orders_from_json = pl.read_ndjson(json_path)
print("\n=== Read back from NDJSON ===")
print(orders_from_json.head(5))

Written NDJSON to: /var/folders/19/_f944hvd0sd_72vpz49q73pw0000gn/T/polars_tutorial_output/orders.ndjson

=== Read back from NDJSON ===
shape: (5, 5)
┌──────────┬─────────────┬────────────┬───────────┬──────────────┐
│ order_id ┆ customer_id ┆ order_date ┆ status    ┆ total_amount │
│ ---      ┆ ---         ┆ ---        ┆ ---       ┆ ---          │
│ i64      ┆ i64         ┆ str        ┆ str       ┆ f64          │
╞══════════╪═════════════╪════════════╪═══════════╪══════════════╡
│ 1001     ┆ 1           ┆ 2023-01-10 ┆ completed ┆ 1349.98      │
│ 1002     ┆ 2           ┆ 2023-01-15 ┆ completed ┆ 599.99       │
│ 1003     ┆ 1           ┆ 2023-02-20 ┆ completed ┆ 539.97       │
│ 1004     ┆ 3           ┆ 2023-03-05 ┆ completed ┆ 1749.98      │
│ 1005     ┆ 4           ┆ 2023-03-18 ┆ cancelled ┆ 89.99        │
└──────────┴─────────────┴────────────┴───────────┴──────────────┘


In [79]:
# WRITE & READ PARQUET (recommended format)
parquet_path = os.path.join(output_dir, "full_details.parquet")
full_details.write_parquet(parquet_path)
print(f"Written Parquet to: {parquet_path}")

# Read with predicate pushdown (lazy scan)
print("\n=== Lazy scan Parquet with filter (predicate pushdown) ===")
electronics = (
    pl
    .scan_parquet(parquet_path)
    .filter(pl.col("category") == "Electronics")
    .select("order_id", "name", "product_name", "line_total")
    .collect()
)
print(electronics.head(5))
print(f"\nRows in Electronics: {electronics.height}")

Written Parquet to: /var/folders/19/_f944hvd0sd_72vpz49q73pw0000gn/T/polars_tutorial_output/full_details.parquet

=== Lazy scan Parquet with filter (predicate pushdown) ===
shape: (5, 4)
┌──────────┬───────────────┬─────────────────────┬────────────┐
│ order_id ┆ name          ┆ product_name        ┆ line_total │
│ ---      ┆ ---           ┆ ---                 ┆ ---        │
│ i64      ┆ str           ┆ str                 ┆ f64        │
╞══════════╪═══════════════╪═════════════════════╪════════════╡
│ 1001     ┆ Alice Johnson ┆ Laptop Pro 15       ┆ 1299.99    │
│ 1001     ┆ Alice Johnson ┆ USB-C Hub           ┆ 49.99      │
│ 1003     ┆ Alice Johnson ┆ Mechanical Keyboard ┆ 89.99      │
│ 1003     ┆ Alice Johnson ┆ Monitor 27"         ┆ 399.99     │
│ 1003     ┆ Alice Johnson ┆ USB-C Hub           ┆ 49.99      │
└──────────┴───────────────┴─────────────────────┴────────────┘

Rows in Electronics: 32


In [80]:
# WRITE & READ IPC (Arrow format — fastest for Polars-to-Polars)
ipc_path = os.path.join(output_dir, "products.arrow")
products.write_ipc(ipc_path)
print(f"Written IPC to: {ipc_path}")

products_from_ipc = pl.read_ipc(ipc_path)
print("\n=== Read back from IPC ===")
print(products_from_ipc.head(5))

Written IPC to: /var/folders/19/_f944hvd0sd_72vpz49q73pw0000gn/T/polars_tutorial_output/products.arrow

=== Read back from IPC ===
shape: (5, 5)
┌────────────┬─────────────────┬─────────────┬─────────┬────────────────┐
│ product_id ┆ product_name    ┆ category    ┆ price   ┆ stock_quantity │
│ ---        ┆ ---             ┆ ---         ┆ ---     ┆ ---            │
│ i64        ┆ str             ┆ str         ┆ f64     ┆ i64            │
╞════════════╪═════════════════╪═════════════╪═════════╪════════════════╡
│ 101        ┆ Laptop Pro 15   ┆ Electronics ┆ 1299.99 ┆ 50             │
│ 102        ┆ Wireless Mouse  ┆ Electronics ┆ 29.99   ┆ 200            │
│ 103        ┆ USB-C Hub       ┆ Electronics ┆ 49.99   ┆ 150            │
│ 104        ┆ Standing Desk   ┆ Furniture   ┆ 599.99  ┆ 30             │
│ 105        ┆ Ergonomic Chair ┆ Furniture   ┆ 449.99  ┆ 45             │
└────────────┴─────────────────┴─────────────┴─────────┴────────────────┘


---
## 12. String & Temporal Operations

Polars provides rich `.str` and `.dt` namespaces for string and date/time operations.

In [81]:
# STRING operations
print("=== String Operations ===")
print(
    customers.select(
        pl.col("name"),
        pl.col("name").str.to_uppercase().alias("upper_name"),
        pl.col("email").str.to_lowercase().alias("lower_email"),
        pl.col("name").str.split(" ").list.first().alias("first_name"),
        pl.col("name").str.split(" ").list.last().alias("last_name"),
        pl.col("name").str.len_chars().alias("name_length"),
        pl.col("email").str.split("@").list.last().alias("email_domain"),
    )
)

=== String Operations ===
shape: (10, 7)
┌──────────────┬──────────────┬──────────────┬────────────┬───────────┬─────────────┬──────────────┐
│ name         ┆ upper_name   ┆ lower_email  ┆ first_name ┆ last_name ┆ name_length ┆ email_domain │
│ ---          ┆ ---          ┆ ---          ┆ ---        ┆ ---       ┆ ---         ┆ ---          │
│ str          ┆ str          ┆ str          ┆ str        ┆ str       ┆ u32         ┆ str          │
╞══════════════╪══════════════╪══════════════╪════════════╪═══════════╪═════════════╪══════════════╡
│ Alice        ┆ ALICE        ┆ alice@email. ┆ Alice      ┆ Johnson   ┆ 13          ┆ email.com    │
│ Johnson      ┆ JOHNSON      ┆ com          ┆            ┆           ┆             ┆              │
│ Bob Smith    ┆ BOB SMITH    ┆ bob@email.co ┆ Bob        ┆ Smith     ┆ 9           ┆ email.com    │
│              ┆              ┆ m            ┆            ┆           ┆             ┆              │
│ Charlie      ┆ CHARLIE      ┆ charlie@emai ┆ Cha

In [82]:
# String matching and extraction
print("=== String Matching ===")
print(
    customers.select(
        pl.col("name"),
        pl.col("name").str.contains("son").alias("contains_son"),
        pl.col("name").str.starts_with("A").alias("starts_with_A"),
        pl.col("email").str.contains(r"^[a-e]", literal=False).alias("email_a_to_e"),
    )
)

=== String Matching ===
shape: (10, 4)
┌───────────────┬──────────────┬───────────────┬──────────────┐
│ name          ┆ contains_son ┆ starts_with_A ┆ email_a_to_e │
│ ---           ┆ ---          ┆ ---           ┆ ---          │
│ str           ┆ bool         ┆ bool          ┆ bool         │
╞═══════════════╪══════════════╪═══════════════╪══════════════╡
│ Alice Johnson ┆ true         ┆ true          ┆ true         │
│ Bob Smith     ┆ false        ┆ false         ┆ true         │
│ Charlie Brown ┆ false        ┆ false         ┆ true         │
│ Diana Ross    ┆ false        ┆ false         ┆ true         │
│ Eve Wilson    ┆ true         ┆ false         ┆ true         │
│ Frank Miller  ┆ false        ┆ false         ┆ false        │
│ Grace Lee     ┆ false        ┆ false         ┆ false        │
│ Henry Davis   ┆ false        ┆ false         ┆ false        │
│ Ivy Chen      ┆ false        ┆ false         ┆ false        │
│ Jack Thompson ┆ true         ┆ false         ┆ false        │
└

In [83]:
# DATE/TIME operations
print("=== Date Operations ===")
print(
    orders.select(
        pl.col("order_id"),
        pl.col("order_date"),
        pl.col("order_date").dt.year().alias("year"),
        pl.col("order_date").dt.month().alias("month"),
        pl.col("order_date").dt.weekday().alias("weekday"),
        pl.col("order_date").dt.quarter().alias("quarter"),
        pl.col("order_date").dt.strftime("%B %Y").alias("formatted"),
        (date(2024, 6, 1) - pl.col("order_date")).dt.total_days().alias("days_ago"),
    ).head(10)
)

=== Date Operations ===
shape: (10, 8)
┌──────────┬────────────┬──────┬───────┬─────────┬─────────┬───────────────┬──────────┐
│ order_id ┆ order_date ┆ year ┆ month ┆ weekday ┆ quarter ┆ formatted     ┆ days_ago │
│ ---      ┆ ---        ┆ ---  ┆ ---   ┆ ---     ┆ ---     ┆ ---           ┆ ---      │
│ i64      ┆ date       ┆ i32  ┆ i8    ┆ i8      ┆ i8      ┆ str           ┆ i64      │
╞══════════╪════════════╪══════╪═══════╪═════════╪═════════╪═══════════════╪══════════╡
│ 1001     ┆ 2023-01-10 ┆ 2023 ┆ 1     ┆ 2       ┆ 1       ┆ January 2023  ┆ 508      │
│ 1002     ┆ 2023-01-15 ┆ 2023 ┆ 1     ┆ 7       ┆ 1       ┆ January 2023  ┆ 503      │
│ 1003     ┆ 2023-02-20 ┆ 2023 ┆ 2     ┆ 1       ┆ 1       ┆ February 2023 ┆ 467      │
│ 1004     ┆ 2023-03-05 ┆ 2023 ┆ 3     ┆ 7       ┆ 1       ┆ March 2023    ┆ 454      │
│ 1005     ┆ 2023-03-18 ┆ 2023 ┆ 3     ┆ 6       ┆ 1       ┆ March 2023    ┆ 441      │
│ 1006     ┆ 2023-04-02 ┆ 2023 ┆ 4     ┆ 7       ┆ 2       ┆ April 2023    ┆ 426 

In [84]:
# Date arithmetic
print("=== Date Arithmetic ===")
print(
    orders.select(
        pl.col("order_id"),
        pl.col("order_date"),
        (pl.col("order_date") + timedelta(days=30)).alias("plus_30_days"),
        pl.col("order_date").dt.month_start().alias("month_start"),
        pl.col("order_date").dt.month_end().alias("month_end"),
    ).head(5)
)

=== Date Arithmetic ===
shape: (5, 5)
┌──────────┬────────────┬──────────────┬─────────────┬────────────┐
│ order_id ┆ order_date ┆ plus_30_days ┆ month_start ┆ month_end  │
│ ---      ┆ ---        ┆ ---          ┆ ---         ┆ ---        │
│ i64      ┆ date       ┆ date         ┆ date        ┆ date       │
╞══════════╪════════════╪══════════════╪═════════════╪════════════╡
│ 1001     ┆ 2023-01-10 ┆ 2023-02-09   ┆ 2023-01-01  ┆ 2023-01-31 │
│ 1002     ┆ 2023-01-15 ┆ 2023-02-14   ┆ 2023-01-01  ┆ 2023-01-31 │
│ 1003     ┆ 2023-02-20 ┆ 2023-03-22   ┆ 2023-02-01  ┆ 2023-02-28 │
│ 1004     ┆ 2023-03-05 ┆ 2023-04-04   ┆ 2023-03-01  ┆ 2023-03-31 │
│ 1005     ┆ 2023-03-18 ┆ 2023-04-17   ┆ 2023-03-01  ┆ 2023-03-31 │
└──────────┴────────────┴──────────────┴─────────────┴────────────┘


---
## 13. Missing Data

Polars distinguishes between `null` (missing) and `NaN` (not-a-number for floats).

In [85]:
# Create sample data with nulls
df_nulls = pl.DataFrame({
    "id": [1, 2, 3, 4, 5],
    "name": ["Alice", "Bob", None, "Diana", "Eve"],
    "amount": [100.0, None, 200.0, None, 150.0],
    "score": [85, 92, None, 78, None],
})

print("=== Original ===")
print(df_nulls)

print("\n=== Null counts ===")
print(df_nulls.null_count())

=== Original ===
shape: (5, 4)
┌─────┬───────┬────────┬───────┐
│ id  ┆ name  ┆ amount ┆ score │
│ --- ┆ ---   ┆ ---    ┆ ---   │
│ i64 ┆ str   ┆ f64    ┆ i64   │
╞═════╪═══════╪════════╪═══════╡
│ 1   ┆ Alice ┆ 100.0  ┆ 85    │
│ 2   ┆ Bob   ┆ null   ┆ 92    │
│ 3   ┆ null  ┆ 200.0  ┆ null  │
│ 4   ┆ Diana ┆ null   ┆ 78    │
│ 5   ┆ Eve   ┆ 150.0  ┆ null  │
└─────┴───────┴────────┴───────┘

=== Null counts ===
shape: (1, 4)
┌─────┬──────┬────────┬───────┐
│ id  ┆ name ┆ amount ┆ score │
│ --- ┆ ---  ┆ ---    ┆ ---   │
│ u32 ┆ u32  ┆ u32    ┆ u32   │
╞═════╪══════╪════════╪═══════╡
│ 0   ┆ 1    ┆ 2      ┆ 2     │
└─────┴──────┴────────┴───────┘


In [86]:
# Filter nulls
print("=== Drop rows with any null ===")
print(df_nulls.drop_nulls())

print("\n=== Drop nulls in specific column ===")
print(df_nulls.drop_nulls(subset=["amount"]))

print("\n=== Filter where amount is not null ===")
print(df_nulls.filter(pl.col("amount").is_not_null()))

=== Drop rows with any null ===
shape: (1, 4)
┌─────┬───────┬────────┬───────┐
│ id  ┆ name  ┆ amount ┆ score │
│ --- ┆ ---   ┆ ---    ┆ ---   │
│ i64 ┆ str   ┆ f64    ┆ i64   │
╞═════╪═══════╪════════╪═══════╡
│ 1   ┆ Alice ┆ 100.0  ┆ 85    │
└─────┴───────┴────────┴───────┘

=== Drop nulls in specific column ===
shape: (3, 4)
┌─────┬───────┬────────┬───────┐
│ id  ┆ name  ┆ amount ┆ score │
│ --- ┆ ---   ┆ ---    ┆ ---   │
│ i64 ┆ str   ┆ f64    ┆ i64   │
╞═════╪═══════╪════════╪═══════╡
│ 1   ┆ Alice ┆ 100.0  ┆ 85    │
│ 3   ┆ null  ┆ 200.0  ┆ null  │
│ 5   ┆ Eve   ┆ 150.0  ┆ null  │
└─────┴───────┴────────┴───────┘

=== Filter where amount is not null ===
shape: (3, 4)
┌─────┬───────┬────────┬───────┐
│ id  ┆ name  ┆ amount ┆ score │
│ --- ┆ ---   ┆ ---    ┆ ---   │
│ i64 ┆ str   ┆ f64    ┆ i64   │
╞═════╪═══════╪════════╪═══════╡
│ 1   ┆ Alice ┆ 100.0  ┆ 85    │
│ 3   ┆ null  ┆ 200.0  ┆ null  │
│ 5   ┆ Eve   ┆ 150.0  ┆ null  │
└─────┴───────┴────────┴───────┘


In [87]:
# Fill nulls
print("=== Fill with literal values ===")
print(
    df_nulls.with_columns(
        pl.col("name").fill_null("Unknown"),
        pl.col("amount").fill_null(0.0),
        pl.col("score").fill_null(pl.col("score").mean()),
    )
)

# Forward/backward fill
print("\n=== Forward fill ===")
print(
    df_nulls.with_columns(
        pl.col("amount").forward_fill().alias("amount_ffill"),
        pl.col("amount").backward_fill().alias("amount_bfill"),
    )
)

# Interpolate
print("\n=== Interpolate ===")
print(
    df_nulls.with_columns(
        pl.col("amount").interpolate().alias("amount_interp"),
    )
)

=== Fill with literal values ===
shape: (5, 4)
┌─────┬─────────┬────────┬───────┐
│ id  ┆ name    ┆ amount ┆ score │
│ --- ┆ ---     ┆ ---    ┆ ---   │
│ i64 ┆ str     ┆ f64    ┆ f64   │
╞═════╪═════════╪════════╪═══════╡
│ 1   ┆ Alice   ┆ 100.0  ┆ 85.0  │
│ 2   ┆ Bob     ┆ 0.0    ┆ 92.0  │
│ 3   ┆ Unknown ┆ 200.0  ┆ 85.0  │
│ 4   ┆ Diana   ┆ 0.0    ┆ 78.0  │
│ 5   ┆ Eve     ┆ 150.0  ┆ 85.0  │
└─────┴─────────┴────────┴───────┘

=== Forward fill ===
shape: (5, 6)
┌─────┬───────┬────────┬───────┬──────────────┬──────────────┐
│ id  ┆ name  ┆ amount ┆ score ┆ amount_ffill ┆ amount_bfill │
│ --- ┆ ---   ┆ ---    ┆ ---   ┆ ---          ┆ ---          │
│ i64 ┆ str   ┆ f64    ┆ i64   ┆ f64          ┆ f64          │
╞═════╪═══════╪════════╪═══════╪══════════════╪══════════════╡
│ 1   ┆ Alice ┆ 100.0  ┆ 85    ┆ 100.0        ┆ 100.0        │
│ 2   ┆ Bob   ┆ null   ┆ 92    ┆ 100.0        ┆ 200.0        │
│ 3   ┆ null  ┆ 200.0  ┆ null  ┆ 200.0        ┆ 200.0        │
│ 4   ┆ Diana ┆ null   ┆ 78 

---
## 14. Reshaping (Pivot, Unpivot, Explode)

Transform between wide and long formats.

In [88]:
# PIVOT — long to wide
print("=== Pivot: Orders by Quarter and Status ===")
print(
    orders
    .with_columns(pl.col("order_date").dt.quarter().alias("quarter"))
    .pivot(on="status", index="quarter", values="order_id", aggregate_function="len")
    .fill_null(0)
    .sort("quarter")
)

=== Pivot: Orders by Quarter and Status ===
shape: (4, 5)
┌─────────┬───────────┬───────────┬──────────┬─────────┐
│ quarter ┆ completed ┆ cancelled ┆ returned ┆ pending │
│ ---     ┆ ---       ┆ ---       ┆ ---      ┆ ---     │
│ i8      ┆ u32       ┆ u32       ┆ u32      ┆ u32     │
╞═════════╪═══════════╪═══════════╪══════════╪═════════╡
│ 1       ┆ 5         ┆ 1         ┆ 0        ┆ 1       │
│ 2       ┆ 4         ┆ 0         ┆ 1        ┆ 0       │
│ 3       ┆ 5         ┆ 1         ┆ 0        ┆ 0       │
│ 4       ┆ 7         ┆ 0         ┆ 0        ┆ 0       │
└─────────┴───────────┴───────────┴──────────┴─────────┘


In [89]:
# UNPIVOT (melt) — wide to long
wide_df = pl.DataFrame({
    "product": ["Laptop", "Mouse", "Keyboard"],
    "Q1_sales": [100, 500, 300],
    "Q2_sales": [120, 480, 350],
    "Q3_sales": [90, 520, 280],
    "Q4_sales": [150, 600, 400],
})

print("=== Wide format ===")
print(wide_df)

print("\n=== Unpivot (long format) ===")
print(
    wide_df.unpivot(
        index="product",
        on=["Q1_sales", "Q2_sales", "Q3_sales", "Q4_sales"],
        variable_name="quarter",
        value_name="sales",
    )
)

=== Wide format ===
shape: (3, 5)
┌──────────┬──────────┬──────────┬──────────┬──────────┐
│ product  ┆ Q1_sales ┆ Q2_sales ┆ Q3_sales ┆ Q4_sales │
│ ---      ┆ ---      ┆ ---      ┆ ---      ┆ ---      │
│ str      ┆ i64      ┆ i64      ┆ i64      ┆ i64      │
╞══════════╪══════════╪══════════╪══════════╪══════════╡
│ Laptop   ┆ 100      ┆ 120      ┆ 90       ┆ 150      │
│ Mouse    ┆ 500      ┆ 480      ┆ 520      ┆ 600      │
│ Keyboard ┆ 300      ┆ 350      ┆ 280      ┆ 400      │
└──────────┴──────────┴──────────┴──────────┴──────────┘

=== Unpivot (long format) ===
shape: (12, 3)
┌──────────┬──────────┬───────┐
│ product  ┆ quarter  ┆ sales │
│ ---      ┆ ---      ┆ ---   │
│ str      ┆ str      ┆ i64   │
╞══════════╪══════════╪═══════╡
│ Laptop   ┆ Q1_sales ┆ 100   │
│ Mouse    ┆ Q1_sales ┆ 500   │
│ Keyboard ┆ Q1_sales ┆ 300   │
│ Laptop   ┆ Q2_sales ┆ 120   │
│ Mouse    ┆ Q2_sales ┆ 480   │
│ Keyboard ┆ Q2_sales ┆ 350   │
│ Laptop   ┆ Q3_sales ┆ 90    │
│ Mouse    ┆ Q3_sales ┆

In [90]:
# EXPLODE — expand list columns into rows
print("=== Explode list column ===")
df_lists = pl.DataFrame({
    "name": ["Alice", "Bob"],
    "hobbies": [["reading", "hiking", "coding"], ["gaming", "cooking"]],
    "scores": [[95, 87, 92], [78, 85]],
})
print(df_lists)

print("\n=== After explode ===")
print(df_lists.explode("hobbies", "scores"))

=== Explode list column ===
shape: (2, 3)
┌───────┬─────────────────────────────────┬──────────────┐
│ name  ┆ hobbies                         ┆ scores       │
│ ---   ┆ ---                             ┆ ---          │
│ str   ┆ list[str]                       ┆ list[i64]    │
╞═══════╪═════════════════════════════════╪══════════════╡
│ Alice ┆ ["reading", "hiking", "coding"] ┆ [95, 87, 92] │
│ Bob   ┆ ["gaming", "cooking"]           ┆ [78, 85]     │
└───────┴─────────────────────────────────┴──────────────┘

=== After explode ===
shape: (5, 3)
┌───────┬─────────┬────────┐
│ name  ┆ hobbies ┆ scores │
│ ---   ┆ ---     ┆ ---    │
│ str   ┆ str     ┆ i64    │
╞═══════╪═════════╪════════╡
│ Alice ┆ reading ┆ 95     │
│ Alice ┆ hiking  ┆ 87     │
│ Alice ┆ coding  ┆ 92     │
│ Bob   ┆ gaming  ┆ 78     │
│ Bob   ┆ cooking ┆ 85     │
└───────┴─────────┴────────┘


/var/folders/19/_f944hvd0sd_72vpz49q73pw0000gn/T/ipykernel_16437/2110806309.py:11: DeprecationWarning: In Polars 2.0, the default behavior for `empty_as_null` will change to `False`. To keep the current behavior, explicitly set `empty_as_null=True`.
  print(df_lists.explode("hobbies", "scores"))


---
## 15. Performance Tips & Patterns

Best practices for getting the most out of Polars.

In [91]:
# TIP 1: Use expressions instead of apply/map — they run in parallel
import time

large_df = pl.DataFrame({
    "value": np.random.randn(1_000_000),
    "group": np.random.choice(["A", "B", "C", "D"], 1_000_000),
})

# Good: native expression (vectorized, parallel)
start = time.time()
_ = large_df.with_columns((pl.col("value") * 2 + 1).alias("transformed"))
expr_time = time.time() - start

# Bad: map_elements (row-by-row Python, single-threaded)
start = time.time()
_ = large_df.with_columns(
    pl
    .col("value")
    .map_elements(lambda x: x * 2 + 1, return_dtype=pl.Float64)
    .alias("transformed")
)
map_time = time.time() - start

print(f"Expression (vectorized): {expr_time * 1000:.1f}ms")
print(f"map_elements (Python):   {map_time * 1000:.1f}ms")
print(f"Speedup: {map_time / expr_time:.0f}x")

Expression (vectorized): 3.4ms
map_elements (Python):   81.8ms
Speedup: 24x


/var/folders/19/_f944hvd0sd_72vpz49q73pw0000gn/T/ipykernel_16437/3957761052.py:19: PolarsInefficientMapWarning: 
Expr.map_elements is significantly slower than the native expressions API.
Only use if you absolutely CANNOT implement your logic otherwise.
Replace this expression...
  - pl.col("value").map_elements(lambda x: ...)
with this one instead:
  + (pl.col("value") * 2) + 1

  .map_elements(lambda x: x * 2 + 1, return_dtype=pl.Float64)


In [92]:
# TIP 2: Use lazy mode for complex queries
# Polars optimizes: predicate pushdown, projection pushdown, slice pushdown

start = time.time()
eager_result = (
    large_df
    .filter(pl.col("group") == "A")
    .with_columns((pl.col("value") ** 2).alias("squared"))
    .group_by("group")
    .agg(pl.col("squared").mean())
)
eager_time = time.time() - start

start = time.time()
lazy_result = (
    large_df
    .lazy()
    .filter(pl.col("group") == "A")
    .with_columns((pl.col("value") ** 2).alias("squared"))
    .group_by("group")
    .agg(pl.col("squared").mean())
    .collect()
)
lazy_time = time.time() - start

print(f"Eager: {eager_time * 1000:.1f}ms")
print(f"Lazy:  {lazy_time * 1000:.1f}ms")
print(f"\nResults match: {eager_result.equals(lazy_result)}")

Eager: 3.4ms
Lazy:  2.6ms

Results match: False


In [93]:
# TIP 3: Use scan_* for large files (reads only what's needed)
# pl.scan_parquet(path)  — lazy, predicate pushdown, projection pushdown
# pl.scan_csv(path)      — lazy, streaming capable
# pl.scan_ipc(path)      — lazy, fastest format

# TIP 4: Use streaming for datasets larger than RAM
# large_lazy_query.collect(streaming=True)

# TIP 5: Prefer Parquet or IPC over CSV
# - Schema is preserved (no type inference needed)
# - Columnar: only reads needed columns
# - Compressed: smaller file sizes
# - Supports predicate pushdown

print("=== File format comparison ===")
csv_size = os.path.getsize(csv_path)
parquet_size = os.path.getsize(parquet_path)
ipc_size = os.path.getsize(ipc_path)
print(f"CSV:     {csv_size:>8,} bytes")
print(f"Parquet: {parquet_size:>8,} bytes")
print(f"IPC:     {ipc_size:>8,} bytes")

=== File format comparison ===
CSV:          666 bytes
Parquet:    4,038 bytes
IPC:        2,237 bytes


In [94]:
# TIP 6: Use sink_* for writing large lazy results without materializing
sink_path = os.path.join(output_dir, "sink_output.parquet")
(
    large_df
    .lazy()
    .filter(pl.col("value") > 0)
    .with_columns((pl.col("value") * 100).round(2).alias("scaled"))
    .sink_parquet(sink_path)
)
print(f"Sink written: {os.path.getsize(sink_path):,} bytes")
print(f"Rows: {pl.scan_parquet(sink_path).select(pl.len()).collect().item()}")

Sink written: 5,478,546 bytes
Rows: 501169


In [95]:
# Cleanup
import shutil

if os.path.exists(output_dir):
    shutil.rmtree(output_dir)
    print(f"Cleaned up: {output_dir}")

print("Tutorial complete!")

Cleaned up: /var/folders/19/_f944hvd0sd_72vpz49q73pw0000gn/T/polars_tutorial_output
Tutorial complete!


---
## Summary

### Key Takeaways

1. **Expressions** are the core API — composable, parallel, optimizable
2. **Lazy mode** (`lazy()` / `scan_*`) enables query optimization
3. **No index** — operations are explicit and predictable
4. **`.over()`** replaces SQL window functions elegantly
5. **Native nested types** (List, Struct) — no need to normalize everything
6. **Parquet/IPC** preferred over CSV for performance
7. **Streaming** (`collect(streaming=True)`, `sink_*`) for larger-than-RAM data
8. **Avoid `map_elements`** — use native expressions for 10-100x speedups
9. **Column selectors** (`cs.*`) for dynamic column selection by type/pattern
10. **Strict typing** catches errors early and enables optimizations

### Polars vs PySpark

| Aspect | Polars | PySpark |
|--------|--------|--------|
| Scale | Single machine (GBs) | Distributed cluster (TBs) |
| Startup | Instant | JVM startup overhead |
| API | Expressions + `.over()` | DataFrame API + SQL |
| Lazy eval | Built-in (`lazy()`) | Always lazy (actions trigger) |
| Performance | Multi-threaded Rust | Distributed JVM |
| Nested data | Native List/Struct | Requires schema definition |

### Next Steps

- Explore **Polars plugins** for custom Rust expressions
- Try **polars-xdt** for extended date/time operations
- Learn **Delta Lake** integration (`pl.read_delta()`)
- Practice with larger datasets (NYC Taxi, TPC-H benchmarks)